# 03 — Vector Databases: From Embeddings Theory to Production Search

## Part 1: Theory

### 1.1 What Are Embeddings?

**Evolution**: One-hot → Word2Vec (2013) → GloVe → ELMo → BERT → Sentence Transformers

Embeddings map text to **dense vectors** where semantic similarity = geometric proximity.

```
"king" - "man" + "woman" ≈ "queen"  (Word2Vec's famous analogy)
```

Modern sentence embeddings capture full meaning: `embed("The cat sat on the mat")` → [0.12, -0.45, 0.78, ...] (384-1536 dims)

### 1.2 Similarity Measures

| Measure | Formula | Range | Best For |
|---------|---------|-------|----------|
| **Cosine** | A·B / (‖A‖×‖B‖) | [-1, 1] | Normalized embeddings (most common) |
| **Dot Product** | A·B | (-∞, +∞) | When magnitude matters |
| **Euclidean (L2)** | √Σ(a_i - b_i)² | [0, +∞) | Spatial distance |
| **Manhattan (L1)** | Σ|a_i - b_i| | [0, +∞) | High-dimensional sparse |

### 1.3 Indexing Algorithms

| Algorithm | Complexity | Recall | Memory | How It Works |
|-----------|-----------|--------|--------|-------------|
| **Brute Force** | O(n×d) | 100% | O(n×d) | Compare query to every vector |
| **LSH** | O(1) avg | ~90% | O(n×d×L) | Hash similar vectors to same bucket |
| **IVF** | O(√n×d) | ~95% | O(n×d) | Cluster vectors, search nearest clusters |
| **HNSW** | O(log n×d) | ~99% | O(n×d×M) | Navigable small-world graph |
| **PQ** | O(n×d/m) | ~85% | O(n×m) | Compress vectors into codes |

### 1.4 How HNSW Works (Most Popular)

```
Layer 2 (sparse):    A ─────────────── D          (long-range connections)
                     │                 │
Layer 1 (medium):    A ── B ─── C ─── D ── E     (medium connections)
                     │    │     │     │    │
Layer 0 (dense):     A─B─C─D─E─F─G─H─I─J─K─L    (short-range, all nodes)
```

**Search**: Start at top layer → greedy walk to nearest → drop to next layer → repeat

**Parameters**: M (connections per node), ef_construction (build quality), ef_search (query quality)

### 1.5 Chunking Strategies

| Strategy | Chunk Size | Overlap | Pros | Cons |
|----------|-----------|---------|------|------|
| Fixed-size | 500 tokens | 50 | Simple, predictable | Splits mid-sentence |
| Recursive | Variable | Variable | Respects structure | More complex |
| Semantic | By meaning | None | Best retrieval quality | Slow, needs model |
| Document-aware | By section | None | Preserves context | Needs structured docs |

### 1.6 Hybrid Search

```
Query → ┌─ Dense Search (semantic meaning) ──────┐
        │                                         ├─ RRF Fusion → Final Ranking
        └─ Sparse Search (BM25 keywords) ─────────┘

RRF Score = Σ 1/(k + rank_i)  where k=60 typically
```

---

## Part 2: Implementation

In [ ]:
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer("all-MiniLM-L6-v2")
EMBEDDING_DIM = 384

# Sample corpus
documents = [
    "Retrieval-augmented generation combines search with LLM generation for grounded answers.",
    "Vector databases store high-dimensional embeddings for fast similarity search.",
    "FAISS is a library for efficient similarity search developed by Meta AI Research.",
    "Pinecone is a managed vector database service with automatic scaling and filtering.",
    "Weaviate is an open-source vector search engine with built-in vectorization modules.",
    "Cosine similarity measures the angle between two vectors in embedding space.",
    "Approximate nearest neighbor algorithms trade perfect recall for massive speed gains.",
    "HNSW builds a hierarchical graph where each layer has exponentially fewer nodes.",
    "Product quantization compresses vectors by splitting them into sub-vectors and codebook lookup.",
    "Embedding models convert text into dense numerical representations capturing semantic meaning.",
    "Chunking strategies significantly affect retrieval quality in RAG pipelines.",
    "Hybrid search combines dense vector search with sparse BM25 keyword matching via reciprocal rank fusion.",
    "Metadata filtering narrows search results by document attributes before or after vector search.",
    "ChromaDB is a lightweight open-source embedding database that runs entirely in-process.",
    "The curse of dimensionality makes brute-force search impractical beyond millions of vectors.",
]

embeddings = embedder.encode(documents)
print(f"Corpus: {len(documents)} docs, dim: {embeddings.shape[1]}")

### FAISS: Multiple Index Types

In [ ]:
import faiss

# Normalize for cosine similarity
embeddings_norm = embeddings.copy()
faiss.normalize_L2(embeddings_norm)

# 1. Flat (brute force) - 100% recall
index_flat = faiss.IndexFlatIP(EMBEDDING_DIM)
index_flat.add(embeddings_norm)

# 2. IVF (inverted file) - approximate
nlist = 4  # number of clusters (use more for larger datasets)
quantizer = faiss.IndexFlatIP(EMBEDDING_DIM)
index_ivf = faiss.IndexIVFFlat(quantizer, EMBEDDING_DIM, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(embeddings_norm)
index_ivf.add(embeddings_norm)
index_ivf.nprobe = 2  # search 2 of 4 clusters

# 3. HNSW - graph-based
index_hnsw = faiss.IndexHNSWFlat(EMBEDDING_DIM, 16)  # M=16 connections
index_hnsw.hnsw.efConstruction = 40
index_hnsw.hnsw.efSearch = 16
index_hnsw.add(embeddings_norm)

def search(index, query, k=5):
    q = embedder.encode([query])
    faiss.normalize_L2(q)
    scores, indices = index.search(q, k)
    return [(documents[i], float(scores[0][j])) for j, i in enumerate(indices[0]) if i >= 0]

print("=== FAISS Flat (exact) ===")
for doc, score in search(index_flat, "How does approximate search work?", k=3):
    print(f"  [{score:.4f}] {doc[:80]}")

print("\n=== FAISS HNSW (approximate) ===")
for doc, score in search(index_hnsw, "How does approximate search work?", k=3):
    print(f"  [{score:.4f}] {doc[:80]}")

### ChromaDB (Fully Local, No Setup)

In [ ]:
import chromadb

client = chromadb.Client()  # in-memory, no server needed
collection = client.create_collection("docs", metadata={"hnsw:space": "cosine"})

# Add documents with metadata
collection.add(
    ids=[f"doc-{i}" for i in range(len(documents))],
    documents=documents,
    metadatas=[{"source": "textbook", "topic": "vector_db" if "vector" in d.lower() else "general"} for d in documents],
)

# Query
results = collection.query(query_texts=["What is HNSW?"], n_results=3)
print("=== ChromaDB Results ===")
for doc, dist in zip(results["documents"][0], results["distances"][0]):
    print(f"  [dist={dist:.4f}] {doc[:80]}")

# Query with metadata filter
results_filtered = collection.query(
    query_texts=["database"], n_results=3,
    where={"topic": "vector_db"}
)
print("\n=== Filtered (topic=vector_db) ===")
for doc in results_filtered["documents"][0]:
    print(f"  {doc[:80]}")

### Benchmark: Index Comparison

In [ ]:
queries = ["How do vector databases work?", "What is cosine similarity?", "Explain chunking for RAG", "HNSW algorithm", "hybrid search"]

def benchmark_index(index, name, queries, runs=20):
    latencies = []
    for _ in range(runs):
        for q in queries:
            q_emb = embedder.encode([q])
            faiss.normalize_L2(q_emb)
            start = time.time()
            index.search(q_emb, 5)
            latencies.append((time.time() - start) * 1000)
    return {"index": name, "avg_ms": np.mean(latencies), "p95_ms": np.percentile(latencies, 95), "qps": 1000/np.mean(latencies)}

bench = pd.DataFrame([
    benchmark_index(index_flat, "Flat (exact)", queries),
    benchmark_index(index_ivf, "IVF (nprobe=2)", queries),
    benchmark_index(index_hnsw, "HNSW (M=16)", queries),
])
print(bench.to_string(index=False))

### Chunking Strategies Comparison

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter

sample_doc = """Vector databases are specialized systems for storing and querying high-dimensional vectors. They use approximate nearest neighbor algorithms like HNSW to achieve sub-millisecond search at scale. The key trade-off is between recall (accuracy) and speed. HNSW provides excellent recall (>99%) with logarithmic search complexity. For production systems, you also need metadata filtering, which allows narrowing results by attributes like date, category, or access level before or after the vector search step."""

# Fixed-size chunking
fixed_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=20, separator=" ")
fixed_chunks = fixed_splitter.split_text(sample_doc)

# Recursive chunking (respects sentence boundaries)
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
recursive_chunks = recursive_splitter.split_text(sample_doc)

print(f"Fixed chunks ({len(fixed_chunks)}):")
for i, c in enumerate(fixed_chunks): print(f"  {i}: '{c[:60]}...'")
print(f"\nRecursive chunks ({len(recursive_chunks)}):")
for i, c in enumerate(recursive_chunks): print(f"  {i}: '{c[:60]}...'")

## Part 3: Comparison Summary

| Feature | FAISS | ChromaDB | Pinecone | Weaviate | Qdrant |
|---------|-------|----------|----------|----------|--------|
| Type | Library | Embedded DB | Managed SaaS | Self-hosted/Cloud | Self-hosted/Cloud |
| Setup | `pip install` | `pip install` | API key | Docker | Docker |
| Scaling | Manual | Single-node | Automatic | Kubernetes | Kubernetes |
| Metadata Filter | ❌ (external) | ✅ | ✅ | ✅ | ✅ |
| Hybrid Search | ❌ | ❌ | ✅ | ✅ | ✅ |
| Persistence | Manual save | SQLite | Cloud | Disk | Disk |
| Best For | Prototyping, batch | Local dev, small apps | Production SaaS | Flexible self-hosted | High-performance |
| Cost | Free | Free | Pay-per-use | Free (self-host) | Free (self-host) |

### Decision Guide
```
Need production managed service? → Pinecone
Need open-source + hybrid search? → Weaviate or Qdrant
Prototyping / notebook? → ChromaDB or FAISS
Maximum raw speed? → FAISS with GPU
```

### Next → 04_rag_pipeline.ipynb